# 第 5 课：物理约束反思

目标：对候选轨迹执行“检查—修正—复查”，拒绝明显违反飞行规律的输出。

In [ ]:
import sys
from pathlib import Path

# 同时兼容：从仓库根目录启动 Jupyter，或从 notebooks/ 目录启动。
search_starts = [Path.cwd(), *Path.cwd().parents]
repo_root = next((path for path in search_starts if (path / "pyproject.toml").exists()), None)
if repo_root is None:
    raise RuntimeError("没有找到 pyproject.toml；请从仓库目录启动 Jupyter。")

src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

print(f"仓库根目录: {repo_root}")
print(f"Python: {sys.version.split()[0]}")

## 5.1 准备历史和非法候选

核心源码：[reflection.py](../src/memcast_uav/reflection.py)  
文字讲解：[05_reflection.md](../tutorial/05_reflection.md)

In [ ]:
import numpy as np

from memcast_uav.data import make_synthetic_flight, make_train_test_windows
from memcast_uav.reflection import (
    FlightConstraints,
    check_constraints,
    reflect_candidate,
    trajectory_metrics,
)

flight = make_synthetic_flight(n_points=360)
_, test = make_train_test_windows(flight, split_index=252)
window = test[0]

candidate = window.future.copy()
candidate[:, 0] += np.linspace(0.0, 300.0, len(candidate))
candidate[:, 2] = np.linspace(window.history[-1, 2], 180.0, len(candidate))
print("非法候选形状:", candidate.shape)

## 5.2 检查违反了哪些约束

In [ ]:
constraints = FlightConstraints(
    max_speed=8.0,
    max_acceleration=1.5,
    max_altitude=120.0,
)
before_issues = check_constraints(
    window.history,
    candidate,
    window.dt,
    constraints,
)
print("修正前问题:", before_issues)
print("修正前指标:", trajectory_metrics(window.history, candidate, window.dt))

## 5.3 运行确定性反思修正

In [ ]:
repaired, trace = reflect_candidate(
    window.history,
    candidate,
    window.dt,
    constraints,
)
after_issues = check_constraints(
    window.history,
    repaired,
    window.dt,
    constraints,
)
print("每轮反思记录:", trace)
print("修正后问题:", after_issues)
print("修正后指标:", trajectory_metrics(window.history, repaired, window.dt))
assert not after_issues

## 5.4 分块练习：一次只改变一个约束

In [ ]:
for max_speed in (8.0, 6.0):
    trial_constraints = FlightConstraints(
        max_speed=max_speed,
        max_acceleration=1.5,
        max_altitude=120.0,
    )
    trial, trial_trace = reflect_candidate(
        window.history,
        candidate,
        window.dt,
        trial_constraints,
    )
    metrics = trajectory_metrics(window.history, trial, window.dt)
    print(
        f"max_speed={max_speed}: 反思轮数={len(trial_trace)-1}, "
        f"最终最大速度={metrics['max_speed']:.3f}"
    )

TODO：设计一个圆形禁飞区 `(center_x, center_y, radius)`。
先考虑如何检查候选点是否落入圆内，再考虑修正策略。

## 5.5 本课验收

In [ ]:
import subprocess

completed = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_reflection.py", "-q"],
    cwd=repo_root,
    check=True,
    text=True,
    capture_output=True,
)
print(completed.stdout)

[← 第 4 课](04_memory.ipynb) · [教程目录](README.md) · [下一课：因果置信度 →](06_confidence.ipynb)